# Controlling Dobot

## lower-level Driver

In [ ]:
from dobot import DobotDriver
import time

In [ ]:
# required packages
#!pip install pyserial numpy matplotlib

In [ ]:
# driver = DobotDriver('COM4')
# fix permissions possibly:
#   sudo chmod o+rw /dev/ttyACM0
driver = DobotDriver('/dev/ttyACM0')
driver.Open()
successes = 0
i = 0
while True:
	ret = driver.isReady()
	if ret[0] and ret[1]:
		successes += 1
	if successes > 10:
		print("Dobot ready!")
		break
	if i > 100:
		raise Exception('Comm problem')

In [ ]:
print('Accelerometer data returned', driver.GetAccelerometers())

In [ ]:
gripper = 480
toolRotation = 0
steps1 = driver.stepsToCmdVal(27)
steps2 = driver.stepsToCmdVal(3)
steps3 = driver.stepsToCmdVal(14)
driver.SetCounters(0, 0, 0)

print("Executing 20 commands with steps 27, -3, -14. Expecting GetCounters() to return:", 27*20, -3*20, -14*20)
errors = 0
for i in range(20):
	ret = (0, 0)
	while not ret[1]:
		ret = driver.Steps(steps1, steps2, steps3, 1, 0, 1, gripper, toolRotation)

time.sleep(3)
print(driver.GetCounters())

In [ ]:
freq = [
	   0,
	  50,
	 250,
	 400,
	 600,
	 800,
	 950,
	1150,
	1300,
	1450,
	1500,
	1800
]

def execute(keys1, keys2, keys3, direction1, direction2, direction3):
	for key1, key2, key3 in zip(keys1, keys2, keys3):
		code1 = driver.freqToCmdVal(key1)
		code2 = driver.freqToCmdVal(key2)
		code3 = driver.freqToCmdVal(key3)
		for i in range(0, 4):
			ret = (1, 0)
			# Check for return from Arduino to make sure the command was queued.
			# See function desciption for more details.
			while not ret[1]:
				ret = driver.Steps(code1, code2, code3, direction1, direction2, direction3, gripper, toolRotation)

increasing = freq
decreasing = sorted(freq, reverse=True)
execute(increasing, [], increasing, 0, 0, 0)
while False:
	execute(decreasing, increasing, decreasing, 0, 0, 0)
	execute(increasing, decreasing, increasing, 1, 0, 1)
	execute(decreasing, increasing, decreasing, 1, 1, 1)
	execute(increasing, decreasing, increasing, 0, 1, 0)

## higher-level SDK

In [ ]:
from dobot import Dobot
dobot = Dobot('/dev/ttyACM0')
dobot.pos

In [ ]:
# Maximum speed in mm/s
speed = 40
# Acceleration in mm/s^2
acceleration = 30
import numpy as np
nextPos = np.array(dobot.pos) + (1,2,3)
nextPos

In [ ]:
nextPos += (0, 0, 0)
dobot.MoveWithSpeed(*nextPos, speed, acceleration)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def circle_points(center, radius, n=100):
    cx, cy = center[:2]
    theta = np.linspace(0, 2*np.pi, n, endpoint=False)
    x = cx + radius * np.cos(theta)
    y = cy + radius * np.sin(theta)
    pts = np.column_stack((x, y))
    # combine and repeat the first point at the end
    return np.vstack((pts, pts[0]))

def plot_points(x, y, center):
    plt.figure(figsize=(5,5))
    plt.plot(x, y, marker='o', markersize=4, linestyle='-')   # circle
    plt.scatter(*center, color='red')                    # center
    plt.gca().set_aspect('equal', 'box')                      # equal scaling
    plt.grid(True)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f"Circle center: {tuple(f"{v:.2f}" for v in center[0:2])}, {radius=})")
    plt.show()


radius = 40
pts = circle_points(nextPos, radius, n=20)
x, y = pts[:,0], pts[:,1]
plot_points(x, y, nextPos)

In [ ]:
zPos = nextPos[2] - 9
for pt in pts:
    print(f"Going to {pt} ..")
    dobot.MoveWithSpeed(*pt, zPos, speed, acceleration)

In [ ]:
# back to start pos (center)
dobot.MoveWithSpeed(*nextPos, speed, acceleration)